In [9]:
import sys
from pathlib import Path


PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == "dualtest_experiments" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "DUALTEST"))

from reference_model import ReferenceModel
from pipeline_functions import (
    load_membership_calibrator,
    make_groq_completion_fn,
    dualtest_pipeline_single,
)

In [10]:
import pandas as pd
from pathlib import Path

DATASET_PATH = PROJECT_ROOT / "dataset" / "metadata" / "booktection_sampled_SAGE.csv"
# ajustá el nombre real del csv si es otro

df_eval = pd.read_csv(DATASET_PATH)

df_eval = df_eval[df_eval["is_original"] == True].copy()
df_eval = df_eval[df_eval["length"] == "medium"].copy()

def load_text_from_file_path(file_path):
    path = PROJECT_ROOT / file_path
    return path.read_text(encoding="utf-8").strip()

df_eval["original_text"] = df_eval["file_path"].apply(load_text_from_file_path)

df_eval.head()

,file_name,file_path,dataset_family,source_dataset,book_id,option,is_original,answer,length,language,label,estimated_membership,text_hash,original_text
0,booktection_05136_A.txt,dataset/raw/booktection/booktection_05136_A.txt,booktection,BookTection,Me_Before_You_-_Jojo_Moyes,A,True,A,medium,english,1,member,dbb9c3b34e663ee952d52abc4139e94f,"I lowered Will’s chair, got him inside, and ma..."
1,booktection_06746_A.txt,dataset/raw/booktection/booktection_06746_A.txt,booktection,BookTection,Things_Fall_Apart_-_Chinua_Achebe,A,True,A,medium,english,1,member,66491912d093a92ea7400594af76bce3,Okonkwo ate the food absent-mindedly. 'She sho...
2,booktection_04306_A.txt,dataset/raw/booktection/booktection_04306_A.txt,booktection,BookTection,Fifty_Shades_Darker_-_E_L_James,A,True,A,medium,english,1,member,7ab777419ee3a9edae41c9c10a153b08,I switch on the desk lamp and focus the light ...
3,booktection_13563_A.txt,dataset/raw/booktection/booktection_13563_A.txt,booktection,BookTection,The_Darkness_Before_Them_-_Matthew_Ward,A,True,A,medium,english,0,non_member,188cd0d9ce07d75131d9ab975ff44caf,“The Eternity King too requires an explanation...
4,booktection_13980_A.txt,dataset/raw/booktection/booktection_13980_A.txt,booktection,BookTection,The_Summer_of_Songbirds_-_Kristy_Woodson_Harvey,A,True,A,medium,english,0,non_member,439610a40112760724c8e3e2d518c4ce,My eyes widened. How could he just walk away f...


In [11]:
model, threshold, threshold_info = load_membership_calibrator(
    PROJECT_ROOT / "results" / "membership_calibrator.joblib",
    PROJECT_ROOT / "results" / "membership_threshold.joblib",
)

In [12]:
reference = ReferenceModel(
    model_name="EleutherAI/pythia-410m",
    device="cpu",
)

Loading weights: 100%|██████████| 292/292 [00:00<00:00, 510.80it/s]


In [13]:
TARGET_MODEL = "llama-3.3-70b-versatile"
#TARGET_MODEL = "llama-3.1-8b-instant"

completion_fn = make_groq_completion_fn(
    api_key="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",
    target_model=TARGET_MODEL,
)

In [14]:
out_path = PROJECT_ROOT / "results" / f"booktection_sage_medium_{TARGET_MODEL.replace('-', '_').replace('.', '_')}.csv"

if out_path.exists():
    df_done = pd.read_csv(out_path)
    done_ids = set(df_done["file_name"].astype(str))
    rows = df_done.to_dict("records")
    print("Retomando:", len(done_ids))
else:
    done_ids = set()
    rows = []

for _, row in df_eval.iterrows():
    sample_id = str(row["file_name"])

    if sample_id in done_ids:
        continue

    try:
        result = dualtest_pipeline_single(
            source_text=row["original_text"],
            completion_fn=completion_fn,
            reference=reference,
            membership_model=model,
            threshold=threshold,
            target_model_name=TARGET_MODEL,
        )

        rows.append({
            "file_name": row["file_name"],
            "book_id": row["book_id"],
            "option": row["option"],
            "answer": row["answer"],
            "length": row["length"],
            "label": row["label"],
            "membership": row["estimated_membership"],
            "text_hash": row["text_hash"],
            **result,
        })

    except Exception as e:
        rows.append({
            "file_name": row["file_name"],
            "book_id": row.get("book_id"),
            "option": row.get("option"),
            "answer": row.get("answer"),
            "length": row.get("length"),
            "label": row.get("label"),
            "membership": row.get("estimated_membership"),
            "text_hash": row.get("text_hash"),
            "error": str(e),
            "target_model": TARGET_MODEL,
        })

    done_ids.add(sample_id)
    pd.DataFrame(rows).to_csv(out_path, index=False)

print("Guardado en:", out_path)

df_results = pd.DataFrame(rows)
df_results.head()

Error API intento 1/8: Connection error.
Error API intento 2/8: Connection error.
Guardado en: c:\Users\isiva\OneDrive\Escritorio\Ingenieria de software\NLP_Proyecto_Final\results\booktection_sage_medium_llama_3_3_70b_versatile.csv


,file_name,book_id,option,answer,length,label,membership,text_hash,target_model,reference_model,...,edit_similarity,p_rlb,p_esb,log_p_esb,membership_probability,membership_threshold,suspicious,prefix,ground_truth,target_completion
0,booktection_05136_A.txt,Me_Before_You_-_Jojo_Moyes,A,A,medium,1,member,dbb9c3b34e663ee952d52abc4139e94f,llama-3.3-70b-versatile,EleutherAI/pythia-410m,...,0.264085,1.0,0.0,-169.012743,0.488275,0.750126,False,"I lowered Will’s chair, got him inside, and ma...",for the time spent out in the cold air. But it...,"than it had been in weeks, since the accident ..."
1,booktection_06746_A.txt,Things_Fall_Apart_-_Chinua_Achebe,A,A,medium,1,member,66491912d093a92ea7400594af76bce3,llama-3.3-70b-versatile,EleutherAI/pythia-410m,...,0.293478,1.0,0.0,-151.033442,0.496509,0.750126,False,Okonkwo ate the food absent-mindedly. 'She sho...,bowl of cool water from the earthen pot in her...,calabash of cold water from the nearby stream....
2,booktection_04306_A.txt,Fifty_Shades_Darker_-_E_L_James,A,A,medium,1,member,7ab777419ee3a9edae41c9c10a153b08,llama-3.3-70b-versatile,EleutherAI/pythia-410m,...,0.274074,1.0,0.0,-158.028053,0.486521,0.750126,False,I switch on the desk lamp and focus the light ...,"mine. I squint at the picture, getting really,...","his, but it's not just the shape of their eyes..."
3,booktection_13563_A.txt,The_Darkness_Before_Them_-_Matthew_Ward,A,A,medium,0,non_member,188cd0d9ce07d75131d9ab975ff44caf,llama-3.3-70b-versatile,EleutherAI/pythia-410m,...,0.257143,1.0,0.0,-173.539950,0.488275,0.750126,False,“The Eternity King too requires an explanation...,given to spontaneity. For all his cultivated d...,"so forthcoming with their intentions, and Dama..."
4,booktection_13980_A.txt,The_Summer_of_Songbirds_-_Kristy_Woodson_Harvey,A,A,medium,0,non_member,439610a40112760724c8e3e2d518c4ce,llama-3.3-70b-versatile,EleutherAI/pythia-410m,...,0.236934,1.0,0.0,-187.030827,0.488275,0.750126,False,My eyes widened. How could he just walk away f...,I crossed my arms. I knew I was deflecting. I ...,"""I understand that, but this is huge,"" I press..."


In [ ]:
df_ok = df_results.copy()

if "error" in df_ok.columns:
    df_ok = df_ok[df_ok["error"].isna()].copy()

df_ok[[
    "file_name",
    "book_id",
    "label",
    "run_length",
    "edit_similarity",
    "log_p_esb",
    "membership_probability",
    "membership_threshold",
    "suspicious",
    "prefix",
    "ground_truth",
    "target_completion",
]].head()



,file_name,book_id,option,answer,length,label,membership,text_hash,target_model,reference_model,...,edit_similarity,p_rlb,p_esb,log_p_esb,membership_probability,membership_threshold,suspicious,prefix,ground_truth,target_completion
14,booktection_14139_A.txt,Tonight_I_Burn_-_Katharine_J_Adams,A,A,medium,0,non_member,9ac03c81b670560a559e8e054f945417,llama-3.3-70b-versatile,EleutherAI/pythia-410m,...,0.340000,4.382112e-09,0.0,-180.550888,0.923077,0.750126,True,Two silver-uniformed palace guards hold the an...,the hilts of their swords. The left sides of t...,"the hilts of their swords, their faces express..."
196,booktection_03687_A.txt,Alice's_Adventures_in_Wonderland_-_Lewis_Carroll,A,A,medium,1,member,491be694c427a1c8f8636e2588edcc19,llama-3.3-70b-versatile,EleutherAI/pythia-410m,...,0.648485,1.000000e+00,0.0,-151.618623,0.923077,0.750126,True,"`I shall be punished for it now, I suppose, by...",out what it was: at first she thought it must ...,`out what it was: at first she thought it must...
6,booktection_05901_A.txt,The_Chronicles_of_Narnia_The_Voyage_of_the_Daw...,A,A,medium,1,member,ed4e15a01059a16b1656342293bb9087,llama-3.3-70b-versatile,EleutherAI/pythia-410m,...,0.386454,8.925838e-10,0.0,-156.535746,0.920981,0.750126,True,“But just as I was going to put my feet into t...,"underneath the first one, and I’ll have to get...","underneath the one I took off, and I'll just h..."
145,booktection_06961_A.txt,Watership_Down_-_Richard_Adams,A,A,medium,1,member,9a39e8d1a81d6834247fbb6345a98e43,llama-3.3-70b-versatile,EleutherAI/pythia-410m,...,0.368564,1.597174e-10,0.0,-159.749342,0.914209,0.750126,True,"After some time, Hazel woke Buckthorn. Then he...",clocks nor books are alive to all manner of kn...,"clocks nor calendars, nor even the concept of ..."
27,booktection_04954_A.txt,Little_Women_-_Louisa_May_Alcott,A,A,medium,1,member,d1a7c0b74eae562bacf105bbefc54a66,llama-3.3-70b-versatile,EleutherAI/pythia-410m,...,0.276836,1.000000e+00,0.0,-226.883088,0.900855,0.750126,True,He can't get into mischief in that little nunn...,then such gay little parties at the great hous...,"now, such sober talks in the quiet library, wh..."
